# Nhóm 10 – Dự đoán tái nhập viện ở bệnh nhân đái tháo đường (Diabetes 130-US)
**Notebook tổng hợp:** cả nhóm dùng file này để thống nhất cách hoạt động của từng task, **từ trên xuống dưới**. Mỗi ô chạy tiếp ô trước và truyền dữ liệu qua **biến** (không đọc/ghi file trung gian), nên chạy `Runtime → Run all` là ra toàn bộ pipeline.

| Giai đoạn | Task | Người | Biến đầu vào | Biến đầu ra |
|---|---|---|---|---|
| 1.1 | Bỏ thuộc tính định danh | Dũng | `raw` | `df` (47 cột), `groups` |
| 1.2 | Thống kê khuyết, bỏ cột khuyết > 50% | Sơn | `df` | `df` (44 cột) |
| 1.3 | Bỏ dòng khuyết < 2%, tạo nhãn, chia 80/20 | Chiến | `df`, `groups` | `X_train_raw`, `X_test_raw`, `y_train_multi/bin`, `y_test_multi/bin` |
| 2.1 | One-Hot (fit trên train) | Dũng | `X_*_raw` | `X_train_oh`, `X_test_oh` |
| 2.2 | PCA 60 thành phần (ma trận tương quan) | Sơn | `X_*_oh` | `X_train_pca`, `X_test_pca`, `eigvals` |
| 2.3 | Chuẩn hóa sau PCA, scree plot, top 5 vector riêng | Chiến | `X_*_pca` | `X_train_pca_sc`, `X_test_pca_sc` |
| 3 | k-NN, Softmax / Cây, RF / MLP, Ensemble | Dũng / Sơn / Chiến | `X_*_pca(_sc)`, `y_*` | `results` |
| 4 | Đánh giá, confusion matrix, ROC, class imbalance | Dũng / Sơn / Chiến | `results` | bảng và biểu đồ |

**Quy ước chung:** mọi bước học từ dữ liệu (One-Hot, PCA, chuẩn hóa) chỉ **fit trên train**, rồi áp dụng sang test. Hạt giống ngẫu nhiên cố định `SEED = 42`.

## 0. Cài đặt

Clone **cả ba nhánh** vào `branches/Dung`, `branches/Son`, `branches/Chien`. Pipeline bên dưới chạy trong `branches/Dung` (nơi có dữ liệu gốc). Hai nhánh còn lại dùng để **đối chiếu** kết quả của pipeline với những gì Sơn và Chiến đã đẩy lên (các ô "Check chéo").

In [ ]:
import os
REPO = "https://github.com/chiensondung/HCMUT-AnIntroductiontoMachineLearningandApplication-Project.git"
BRANCHES = ["Dung", "Son", "Chien"]
!rm -rf branches
for b in BRANCHES:
    !git clone -q -b {b} {REPO} branches/{b}
BR = os.path.abspath("branches")       # thư mục chứa 3 nhánh
%cd branches/Dung
!pip -q install -r requirements.txt
!ls {BR}

In [ ]:
# Tham số dùng chung cho cả nhóm. Đổi ở đây, không đổi rải rác trong các ô sau.
SEED = 42
MISS_THRESHOLD = 0.50   # bỏ cột khuyết > 50%
ROW_THRESHOLD = 0.02    # bỏ dòng ở các cột khuyết < 2%
N_COMPONENTS = 60       # số thành phần PCA
results = {}            # mỗi mô hình ghi kết quả vào đây, bảng so sánh ở Giai đoạn 4

In [ ]:
def check(name, mine, theirs):
    print(("OK   " if mine == theirs else "LỆCH ") + f"{name}: notebook = {mine} | nhánh = {theirs}")

## Giai đoạn 1: Khảo sát và xử lý dữ liệu khuyết
### 1.1 Bỏ thuộc tính định danh *(Dũng)*
`encounter_id`, `patient_nbr`, `payer_code` không mang ý nghĩa dự đoán. **Lưu ý thống nhất:** `patient_nbr` vẫn cần để chia tập theo bệnh nhân ở bước 1.3, nên được giữ riêng trong biến `groups` (không dùng làm đặc trưng).

In [ ]:
import numpy as np
import pandas as pd

raw = pd.read_csv("data/diabetic_data.csv", na_values="?", low_memory=False)
print("Kích thước gốc:", raw.shape)            # (101766, 50)

ID_COLS = ["encounter_id", "patient_nbr", "payer_code"]
print(raw[ID_COLS].nunique().to_dict())
print("payer_code khuyết (%):", round(raw["payer_code"].isna().mean() * 100, 2))

groups = raw["patient_nbr"].copy()             # chỉ để nhóm bệnh nhân khi chia tập
df = raw.drop(columns=ID_COLS)
assert df.shape == (101766, 47) and not set(ID_COLS) & set(df.columns)
print("Sau khi bỏ định danh:", df.shape)       # (101766, 47)

### 1.2 Thống kê khuyết, bỏ cột khuyết > 50% *(Sơn)*

In [ ]:
miss = df.isna().mean()
print((miss[miss > 0].sort_values(ascending=False) * 100).round(2))

drop_cols = miss[miss > MISS_THRESHOLD].index.tolist()
df = df.drop(columns=drop_cols)
print("Cột bị bỏ:", drop_cols)
print("Kích thước:", df.shape)                  # (101766, 44)
assert df.shape == (101766, 44)

### 1.3 Bỏ dòng khuyết < 2%, tạo nhãn, chia train/test 80/20 *(Chiến)*
Nhãn đa lớp `y_multi` (`<30`, `>30`, `NO`) và nhị phân `y_bin` (1 = tái nhập viện). Chia 80/20 phân tầng theo `y_multi`, **không để một bệnh nhân xuất hiện ở cả hai tập** (nhóm theo `patient_nbr`).

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

miss = df.isna().mean()
low_miss_cols = miss[(miss > 0) & (miss < ROW_THRESHOLD)].index.tolist()
print("Cột khuyết < 2%:", low_miss_cols)

keep = df[low_miss_cols].notna().all(axis=1)
print("Bỏ", int((~keep).sum()), "dòng, còn", int(keep.sum()))
df = df[keep].reset_index(drop=True)
groups = groups[keep].reset_index(drop=True)

y_multi = df.pop("readmitted")
y_bin = (y_multi != "NO").astype(int)

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr, te = next(splitter.split(df, y_multi, groups))
assert not set(groups[tr]) & set(groups[te])    # không rò rỉ bệnh nhân

X_train_raw, X_test_raw = df.iloc[tr].reset_index(drop=True), df.iloc[te].reset_index(drop=True)
y_train_multi, y_test_multi = y_multi.iloc[tr].reset_index(drop=True), y_multi.iloc[te].reset_index(drop=True)
y_train_bin, y_test_bin = y_bin.iloc[tr].reset_index(drop=True), y_bin.iloc[te].reset_index(drop=True)

print("Train:", X_train_raw.shape, "| Test:", X_test_raw.shape)
for name, s in (("train", y_train_multi), ("test", y_test_multi)):
    print(name, s.value_counts(normalize=True).round(3).to_dict())

**Check chéo với Chiến** (`task_1-3-result` trên nhánh `Chien`): số dòng train/test phải bằng nhau.

In [ ]:
ct = pd.read_csv(f"{BR}/Chien/task_1-3-result/train_task1_3.csv", usecols=["y_multi"])
ce = pd.read_csv(f"{BR}/Chien/task_1-3-result/test_task1_3.csv", usecols=["y_multi"])
check("số dòng train", len(y_train_multi), len(ct)); check("số dòng test", len(y_test_multi), len(ce))
check("nhãn train", y_train_multi.value_counts().to_dict(), ct["y_multi"].value_counts().to_dict())

## Giai đoạn 2: One-Hot, PCA, chuẩn hóa
### 2.1 One-Hot Encoding *(Dũng)*
Mã hóa mọi biến phân loại (gồm mã ICD-9 `diag_1..3`, và 3 cột mã số `admission_type_id`, `discharge_disposition_id`, `admission_source_id`). Giá trị khuyết là một loại riêng `Missing`. **Fit trên train**, `handle_unknown="ignore"` cho các loại chỉ xuất hiện ở test.

In [ ]:
from scipy import sparse
from sklearn.preprocessing import OneHotEncoder

ID_AS_CAT = ["admission_type_id", "discharge_disposition_id", "admission_source_id"]
cat_cols = X_train_raw.select_dtypes(exclude="number").columns.tolist() + ID_AS_CAT
num_cols = [c for c in X_train_raw.columns if c not in cat_cols]
print("Biến phân loại:", len(cat_cols), "| biến số:", len(num_cols))

for part in (X_train_raw, X_test_raw):
    part[cat_cols] = part[cat_cols].astype("string").fillna("Missing")

enc = OneHotEncoder(sparse_output=True, dtype="float64", handle_unknown="ignore")
Xc_train = enc.fit_transform(X_train_raw[cat_cols])
Xc_test = enc.transform(X_test_raw[cat_cols])

X_train_oh = sparse.hstack([sparse.csr_matrix(X_train_raw[num_cols].to_numpy("float64")), Xc_train]).tocsr()
X_test_oh = sparse.hstack([sparse.csr_matrix(X_test_raw[num_cols].to_numpy("float64")), Xc_test]).tocsr()
oh_names = num_cols + enc.get_feature_names_out(cat_cols).tolist()
assert X_train_oh.shape[1] == len(oh_names) == X_test_oh.shape[1]
print("One-Hot:", X_train_oh.shape, X_test_oh.shape)
for d in ("diag_1", "diag_2", "diag_3"):
    print(d, "->", sum(n.startswith(d + "_") for n in oh_names), "cột")

### 2.2 PCA 60 thành phần trên ma trận tương quan *(Sơn)*
Tự cài đặt: ma trận tương quan của train → trị riêng/vector riêng → chiếu lên 60 vector riêng lớn nhất. Cột phương sai 0 trên train bị bỏ trước. Tính trực tiếp từ ma trận thưa (không chuyển sang ma trận dày).

In [ ]:
n = X_train_oh.shape[0]
mu = np.asarray(X_train_oh.mean(axis=0)).ravel()
var = np.asarray(X_train_oh.multiply(X_train_oh).mean(axis=0)).ravel() - mu**2

keep_cols = var > 1e-12
X_tr, X_te = X_train_oh[:, keep_cols], X_test_oh[:, keep_cols]
mu, sigma = mu[keep_cols], np.sqrt(var[keep_cols])
pca_names = [c for c, k in zip(oh_names, keep_cols) if k]
print("Bỏ", int((~keep_cols).sum()), "cột phương sai 0 | còn", X_tr.shape[1], "chiều")

cov = (X_tr.T @ X_tr).toarray() / n - np.outer(mu, mu)
corr = cov / np.outer(sigma, sigma)
corr = (corr + corr.T) / 2

eigvals, eigvecs = np.linalg.eigh(corr)
order = np.argsort(eigvals)[::-1]
eigvals, eigvecs = np.clip(eigvals[order], 0, None), eigvecs[:, order]
V = eigvecs[:, :N_COMPONENTS]

def project(X):  # ((X - mu) / sigma) @ V, không làm dày X
    return X @ (V / sigma[:, None]) - (mu / sigma) @ V

X_train_pca, X_test_pca = project(X_tr), project(X_te)
explained = eigvals[:N_COMPONENTS].sum() / eigvals.sum()
print("PCA:", X_train_pca.shape, X_test_pca.shape, "| phương sai giải thích:", round(explained, 4))

**Check chéo với Sơn** (`output/` trên nhánh `Son`): kích thước ma trận PCA và tỉ lệ phương sai giải thích phải khớp.

In [ ]:
son_tr = np.load(f"{BR}/Son/output/X_train_pca.npy", mmap_mode="r")
son_var = pd.read_csv(f"{BR}/Son/output/pca_explained_variance.csv")["ty_le_luy_tich"].iloc[-1]
check("kích thước train PCA", X_train_pca.shape, son_tr.shape)
check("phương sai giải thích (60 thành phần)", round(float(explained), 4), round(float(son_var), 4))

### 2.3 Chuẩn hóa sau PCA, biểu đồ Eigenvalues, top 5 vector riêng *(Chiến)*

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train_pca)           # fit trên train
X_train_pca_sc, X_test_pca_sc = scaler.transform(X_train_pca), scaler.transform(X_test_pca)
print("Độ lệch chuẩn trước chuẩn hóa (min, max):", X_train_pca.std(0).min().round(2), X_train_pca.std(0).max().round(2))
print("Độ lệch chuẩn sau chuẩn hóa  (min, max):", X_train_pca_sc.std(0).min().round(2), X_train_pca_sc.std(0).max().round(2))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(range(1, 101), eigvals[:100], marker="o", ms=3); ax[0].set(title="Eigenvalues (top 100)", xlabel="Thành phần", ylabel="Trị riêng")
ax[1].plot(range(1, N_COMPONENTS + 1), np.cumsum(eigvals[:N_COMPONENTS]) / eigvals.sum()); ax[1].set(title="Phương sai tích lũy", xlabel="Thành phần")
plt.tight_layout(); plt.show()

for k in range(5):  # 5 đặc trưng gốc đóng góp lớn nhất cho mỗi vector riêng
    top = np.argsort(np.abs(V[:, k]))[::-1][:5]
    print(f"PC{k + 1} (trị riêng {eigvals[k]:.2f}):", [(pca_names[i], round(float(V[i, k]), 3)) for i in top])

## Giai đoạn 3: Huấn luyện mô hình
Mỗi người ghi kết quả vào `results[tên_mô_hình]` (xem bảng ở Giai đoạn 4). Nhãn đa lớp: `y_*_multi`; nhị phân: `y_*_bin`.

### 3.1 Softmax Regression *(Dũng)*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

print("Baseline (luôn đoán NO):", round((y_test_multi == "NO").mean(), 4))

grid = GridSearchCV(LogisticRegression(max_iter=1000), {"C": [0.001, 0.01, 0.1, 1, 10]},
                    cv=5, scoring="accuracy", n_jobs=-1).fit(X_train_pca, y_train_multi)
print("CV theo C:", grid.cv_results_["mean_test_score"].round(4), "| tốt nhất:", grid.best_params_)

softmax = grid.best_estimator_
results["Softmax (multi)"] = {"model": softmax, "acc": accuracy_score(y_test_multi, softmax.predict(X_test_pca))}
print("Accuracy test:", round(results["Softmax (multi)"]["acc"], 4))

### 3.2 k-NN: chưa chuẩn hóa so với đã chuẩn hóa *(Dũng)*

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

K_GRID = {"n_neighbors": [51, 101, 201, 301]}
for tag, (Xtr, Xte) in {"chưa chuẩn hóa": (X_train_pca, X_test_pca),
                        "đã chuẩn hóa": (X_train_pca_sc, X_test_pca_sc)}.items():
    g = GridSearchCV(KNeighborsClassifier(), K_GRID, cv=3, scoring="accuracy", n_jobs=-1).fit(Xtr, y_train_multi)
    acc = accuracy_score(y_test_multi, g.predict(Xte))
    results[f"k-NN {tag} (multi)"] = {"model": g.best_estimator_, "acc": acc}
    print(f"k-NN {tag}: CV theo k {g.cv_results_['mean_test_score'].round(4)} | k = {g.best_params_['n_neighbors']} | test {acc:.4f}")

### 3.3 Decision Tree, Random Forest *(Sơn)* — chưa làm
Đầu vào: `X_train_pca`, `y_train_multi` (và `y_train_bin`). Tinh chỉnh `max_depth`, `n_estimators` để chống overfitting. Ghi `results["Decision Tree (multi)"]`, `results["Random Forest (multi)"]` với khóa `model` và `acc`.

In [ ]:
# TODO (Sơn): Decision Tree và Random Forest

### 3.4 MLP (120, 60, 20, ReLU) và Ensemble trọng số AUC-ROC *(Chiến)* — chưa làm
Đầu vào: `X_train_pca_sc` (MLP cần dữ liệu đã chuẩn hóa). Ghi `results["MLP (multi)"]`, `results["Ensemble (multi)"]`.

In [ ]:
# TODO (Chiến): MLP và Ensemble

## Giai đoạn 4: Đánh giá tổng hợp
Bảng so sánh Test Accuracy của mọi mô hình trong `results` (các task còn trống tự được bỏ qua).

In [ ]:
summary = pd.DataFrame({k: {"Test accuracy": round(v["acc"], 4)} for k, v in results.items()}).T
print(summary.sort_values("Test accuracy", ascending=False))

### 4.1 Confusion matrix đa lớp *(Dũng)* — chưa làm
### 4.2 Accuracy nhị phân và đường cong ROC/AUC *(Sơn)* — chưa làm
### 4.3 Ảnh hưởng của Class Imbalance, Random Forest trên dữ liệu chưa PCA *(Chiến)* — chưa làm

In [ ]:
# TODO (Dũng, Sơn, Chiến): Giai đoạn 4